In [0]:

from pyspark.sql import functions as F

measurements_raw = spark.table(
    "workspace.default.bronze_secom_measurements_raw"
)

labels_raw = spark.table(
    "workspace.default.bronze_secom_labels_raw"
)

print("Measurement rows:", measurements_raw.count())
print("Label rows:", labels_raw.count())


Measurement rows: 1567
Label rows: 1567


In [0]:

# Inspect the first raw measurement record

first_row = measurements_raw.orderBy("row_id").first()

values = first_row["raw_line"].split()

print("Row ID:", first_row["row_id"])
print("Number of measurements:", len(values))
print("First 10 measurements:", values[:10])


Row ID: 1
Number of measurements: 590
First 10 measurements: ['3030.93', '2564', '2187.7333', '1411.1265', '1.3602', '100', '97.6133', '0.1242', '1.5005', '0.0162']


In [0]:

# Convert raw measurement strings into 590 numeric columns

feature_names = [f"feature_{i:03d}" for i in range(1, 591)]

parsed_rows = []

for row in measurements_raw.orderBy("row_id").toLocalIterator():
    values = row["raw_line"].split()

    parsed_values = [
        None if value == "?" else float(value)
        for value in values
    ]

    parsed_rows.append((row["row_id"], *parsed_values))

print("Parsed records:", len(parsed_rows))
print("Features per record:", len(feature_names))


Parsed records: 1567
Features per record: 590


In [0]:

from pyspark.sql.types import StructType, StructField, LongType, DoubleType

measurement_schema = StructType(
    [StructField("row_id", LongType(), False)] +
    [StructField(name, DoubleType(), True) for name in feature_names]
)

measurements_df = spark.createDataFrame(
    parsed_rows,
    schema=measurement_schema
)

print("Rows:", measurements_df.count())
print("Columns:", len(measurements_df.columns))

display(measurements_df.limit(5))


Rows: 1567
Columns: 591


row_id,feature_001,feature_002,feature_003,feature_004,feature_005,feature_006,feature_007,feature_008,feature_009,feature_010,feature_011,feature_012,feature_013,feature_014,feature_015,feature_016,feature_017,feature_018,feature_019,feature_020,feature_021,feature_022,feature_023,feature_024,feature_025,feature_026,feature_027,feature_028,feature_029,feature_030,feature_031,feature_032,feature_033,feature_034,feature_035,feature_036,feature_037,feature_038,feature_039,feature_040,feature_041,feature_042,feature_043,feature_044,feature_045,feature_046,feature_047,feature_048,feature_049,feature_050,feature_051,feature_052,feature_053,feature_054,feature_055,feature_056,feature_057,feature_058,feature_059,feature_060,feature_061,feature_062,feature_063,feature_064,feature_065,feature_066,feature_067,feature_068,feature_069,feature_070,feature_071,feature_072,feature_073,feature_074,feature_075,feature_076,feature_077,feature_078,feature_079,feature_080,feature_081,feature_082,feature_083,feature_084,feature_085,feature_086,feature_087,feature_088,feature_089,feature_090,feature_091,feature_092,feature_093,feature_094,feature_095,feature_096,feature_097,feature_098,feature_099,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,feature_166,feature_167,feature_168,feature_169,feature_170,feature_171,feature_172,feature_173,feature_174,feature_175,feature_176,feature_177,feature_178,feature_179,feature_180,feature_181,feature_182,feature_183,feature_184,feature_185,feature_186,feature_187,feature_188,feature_189,feature_190,feature_191,feature_192,feature_193,feature_194,feature_195,feature_196,feature_197,feature_198,feature_199,feature_200,feature_201,feature_202,feature_203,feature_204,feature_205,feature_206,feature_207,feature_208,feature_209,feature_210,feature_211,feature_212,feature_213,feature_214,feature_215,feature_216,feature_217,feature_218,feature_219,feature_220,feature_221,feature_222,feature_223,feature_224,feature_225,feature_226,feature_227,feature_228,feature_229,feature_230,feature_231,feature_232,feature_233,feature_234,feature_235,feature_236,feature_237,feature_238,feature_239,feature_240,feature_241,feature_242,feature_243,feature_244,feature_245,feature_246,feature_247,feature_248,feature_249,feature_250,feature_251,feature_252,feature_253,feature_254,feature_255,feature_256,feature_257,feature_258,feature_259,feature_260,feature_261,feature_262,feature_263,feature_264,feature_265,feature_266,feature_267,feature_268,feature_269,feature_270,feature_271,feature_272,feature_273,feature_274,feature_275,feature_276,feature_277,feature_278,feature_279,feature_280,feature_281,feature_282,feature_283,feature_284,feature_285,feature_286,feature_287,feature_288,feature_289,feature_290,feature_291,feature_292,feature_293,feature_294,feature_295,feature_296,feature_297,feature_298,feature_299,feature_300,feature_301,feature_302,feature_303,feature_304,feature_305,feature_306,feature_307,feature_308,feature_309,feature_310,feature_311,feature_312,feature_313,feature_314,feature_315,feature_316,feature_317,feature_318,feature_319,feature_320,feature_321,feature_322,feature_323,feature_324,feature_325,feature_326,feature_327,feature_328,feature_329,feature_330,feature_331,feature_332,feature_3

In [0]:

# Extract the quality labels from the Bronze table

label_rows = []

for row in labels_raw.orderBy("row_id").toLocalIterator():
    values = row["raw_line"].split()
    label_rows.append((row["row_id"], int(values[0])))

print("Parsed label records:", len(label_rows))
print("First 10 labels:", [r[1] for r in label_rows[:10]])


Parsed label records: 1567
First 10 labels: [-1, -1, 1, -1, -1, -1, -1, -1, -1, -1]


In [0]:

from pyspark.sql.types import IntegerType

labels_schema = StructType([
    StructField("row_id", LongType(), False),
    StructField("quality_label", IntegerType(), False)
])

labels_df = spark.createDataFrame(
    label_rows,
    schema=labels_schema
)

print("Label rows:", labels_df.count())

display(labels_df.limit(10))


Label rows: 1567


row_id,quality_label
1,-1
2,-1
3,1
4,-1
5,-1
6,-1
7,-1
8,-1
9,-1
10,-1


In [0]:

# Combine manufacturing measurements with quality labels

combined_df = measurements_df.join(
    labels_df,
    on="row_id",
    how="inner"
)

print("Combined rows:", combined_df.count())
print("Measurement features:", len(feature_names))
print("Total columns:", len(combined_df.columns))

display(combined_df.select(
    "row_id", "feature_001", "feature_002", "quality_label"
).limit(10))


Combined rows: 1567
Measurement features: 590
Total columns: 592


row_id,feature_001,feature_002,quality_label
1,3030.93,2564.0,-1
2,3095.78,2465.14,-1
3,2932.61,2559.94,1
4,2988.72,2479.9,-1
5,3032.24,2502.87,-1
6,2946.25,2432.84,-1
7,3030.27,2430.12,-1
8,3058.88,2690.15,-1
9,2967.68,2600.47,-1
10,3016.11,2428.37,-1


In [0]:

# Check the distribution of passing and failing products

display(
    combined_df
    .groupBy("quality_label")
    .count()
    .orderBy("quality_label")
)


quality_label,count
-1,1463
1,104


In [0]:

# Count missing values in each measurement feature

missing_counts = combined_df.select([
    F.sum(
        F.col(c).isNull().cast("long")
    ).alias(c)
    for c in feature_names
])

display(missing_counts)


feature_001,feature_002,feature_003,feature_004,feature_005,feature_006,feature_007,feature_008,feature_009,feature_010,feature_011,feature_012,feature_013,feature_014,feature_015,feature_016,feature_017,feature_018,feature_019,feature_020,feature_021,feature_022,feature_023,feature_024,feature_025,feature_026,feature_027,feature_028,feature_029,feature_030,feature_031,feature_032,feature_033,feature_034,feature_035,feature_036,feature_037,feature_038,feature_039,feature_040,feature_041,feature_042,feature_043,feature_044,feature_045,feature_046,feature_047,feature_048,feature_049,feature_050,feature_051,feature_052,feature_053,feature_054,feature_055,feature_056,feature_057,feature_058,feature_059,feature_060,feature_061,feature_062,feature_063,feature_064,feature_065,feature_066,feature_067,feature_068,feature_069,feature_070,feature_071,feature_072,feature_073,feature_074,feature_075,feature_076,feature_077,feature_078,feature_079,feature_080,feature_081,feature_082,feature_083,feature_084,feature_085,feature_086,feature_087,feature_088,feature_089,feature_090,feature_091,feature_092,feature_093,feature_094,feature_095,feature_096,feature_097,feature_098,feature_099,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,feature_166,feature_167,feature_168,feature_169,feature_170,feature_171,feature_172,feature_173,feature_174,feature_175,feature_176,feature_177,feature_178,feature_179,feature_180,feature_181,feature_182,feature_183,feature_184,feature_185,feature_186,feature_187,feature_188,feature_189,feature_190,feature_191,feature_192,feature_193,feature_194,feature_195,feature_196,feature_197,feature_198,feature_199,feature_200,feature_201,feature_202,feature_203,feature_204,feature_205,feature_206,feature_207,feature_208,feature_209,feature_210,feature_211,feature_212,feature_213,feature_214,feature_215,feature_216,feature_217,feature_218,feature_219,feature_220,feature_221,feature_222,feature_223,feature_224,feature_225,feature_226,feature_227,feature_228,feature_229,feature_230,feature_231,feature_232,feature_233,feature_234,feature_235,feature_236,feature_237,feature_238,feature_239,feature_240,feature_241,feature_242,feature_243,feature_244,feature_245,feature_246,feature_247,feature_248,feature_249,feature_250,feature_251,feature_252,feature_253,feature_254,feature_255,feature_256,feature_257,feature_258,feature_259,feature_260,feature_261,feature_262,feature_263,feature_264,feature_265,feature_266,feature_267,feature_268,feature_269,feature_270,feature_271,feature_272,feature_273,feature_274,feature_275,feature_276,feature_277,feature_278,feature_279,feature_280,feature_281,feature_282,feature_283,feature_284,feature_285,feature_286,feature_287,feature_288,feature_289,feature_290,feature_291,feature_292,feature_293,feature_294,feature_295,feature_296,feature_297,feature_298,feature_299,feature_300,feature_301,feature_302,feature_303,feature_304,feature_305,feature_306,feature_307,feature_308,feature_309,feature_310,feature_311,feature_312,feature_313,feature_314,feature_315,feature_316,feature_317,feature_318,feature_319,feature_320,feature_321,feature_322,feature_323,feature_324,feature_325,feature_326,feature_327,feature_328,feature_329,feature_330,feature_331,feature_332,feature_333,feat

In [0]:

# Summarize missing values across all measurement features

missing_dict = missing_counts.first().asDict()

missing_summary = [
    (name, int(count or 0))
    for name, count in missing_dict.items()
]

missing_df = spark.createDataFrame(
    missing_summary,
    ["feature", "missing_count"]
)

print("Total features:", len(missing_summary))
print(
    "Features with missing values:",
    sum(1 for _, count in missing_summary if count > 0)
)
print(
    "Features completely missing:",
    sum(1 for _, count in missing_summary if count == 1567)
)

display(
    missing_df
    .filter(F.col("missing_count") > 0)
    .orderBy(F.desc("missing_count"))
)


Total features: 590
Features with missing values: 0
Features completely missing: 0


feature,missing_count


In [0]:

# Check missing values directly in the parsed data

print(
    "Missing values in parsed measurements:",
    measurements_df.select(
        F.sum(
            F.col("feature_001").isNull().cast("long")
        ).alias("missing_feature_001")
    ).first()["missing_feature_001"]
)

print(
    "Question marks in first raw record:",
    measurements_raw.orderBy("row_id")
    .first()["raw_line"].count("?")
)


Missing values in parsed measurements: 0
Question marks in first raw record: 0


In [0]:

# Verify missing values across every feature

null_checks = [
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in feature_names
]

null_results = measurements_df.select(*null_checks).first().asDict()

total_missing = sum(int(v or 0) for v in null_results.values())
features_with_missing = sum(1 for v in null_results.values() if v > 0)

print("Total missing values:", total_missing)
print("Features with missing values:", features_with_missing)


Total missing values: 0
Features with missing values: 0


In [0]:

# Check for NaN values across all measurement features

nan_checks = [
    F.sum(F.isnan(F.col(c)).cast("long")).alias(c)
    for c in feature_names
]

nan_results = measurements_df.select(*nan_checks).first().asDict()

total_nan = sum(int(v or 0) for v in nan_results.values())

print("Total NaN values:", total_nan)


Total NaN values: 41951


In [0]:

# Convert NaN values to null for all measurement features

for column_name in feature_names:
    measurements_df = measurements_df.withColumn(
        column_name,
        F.when(
            F.isnan(F.col(column_name)),
            None
        ).otherwise(F.col(column_name))
    )

print("NaN-to-null conversion completed.")


NaN-to-null conversion completed.


In [0]:

# Efficiently count all missing values in one scan

missing_counts = measurements_df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in feature_names
]).first().asDict()

total_missing = sum(int(v or 0) for v in missing_counts.values())

print("Total missing values:", total_missing)


Total missing values: 41951


In [0]:

# Summarize missing features

counts = list(missing_counts.values())

print("Total features:", len(counts))
print("Features with missing values:", sum(v > 0 for v in counts))
print("Completely missing features:", sum(v == 1567 for v in counts))
print("Features over 50% missing:", sum(v > 783 for v in counts))


Total features: 590
Features with missing values: 538
Completely missing features: 0
Features over 50% missing: 28


In [0]:

# Remove features with more than 50% missing values

columns_to_drop = [
    name for name, count in missing_counts.items()
    if count > 783
]

clean_feature_names = [
    name for name in feature_names
    if name not in columns_to_drop
]

measurements_df = measurements_df.drop(*columns_to_drop)

print("Features removed:", len(columns_to_drop))
print("Features remaining:", len(clean_feature_names))
print("Rows retained:", measurements_df.count())


Features removed: 28
Features remaining: 562
Rows retained: 1567


In [0]:

# Rebuild the cleaned features and fill missing values in one operation

clean_feature_names = [
    c for c in feature_names if c not in columns_to_drop
]

base_df = spark.createDataFrame(parsed_rows, schema=measurement_schema)

measurements_df = base_df.select(
    "row_id",
    *[
        F.when(F.isnan(F.col(c)), None).otherwise(F.col(c)).alias(c)
        for c in clean_feature_names
    ]
)

medians = measurements_df.approxQuantile(
    clean_feature_names, [0.5], 0.01
)

fill_values = {
    name: values[0]
    for name, values in zip(clean_feature_names, medians)
    if values
}

measurements_df = measurements_df.fillna(fill_values)

print("Features retained:", len(clean_feature_names))
print("Median imputation prepared.")


Features retained: 562
Median imputation prepared.


In [0]:

# Verify that no missing values remain

check = measurements_df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in clean_feature_names
]).first().asDict()

print("Remaining missing values:", sum(int(v or 0) for v in check.values()))
print("Rows:", measurements_df.count())
print("Features:", len(clean_feature_names))


Remaining missing values: 0
Rows: 1567
Features: 562


In [0]:

# Join cleaned measurements with quality labels

silver_df = measurements_df.join(
    labels_df,
    on="row_id",
    how="inner"
)

print("Rows:", silver_df.count())
print("Total columns:", len(silver_df.columns))

display(silver_df.select("row_id", "quality_label").limit(10))


Rows: 1567
Total columns: 564


row_id,quality_label
1,-1
2,-1
3,1
4,-1
5,-1
6,-1
7,-1
8,-1
9,-1
10,-1


In [0]:

# Save cleaned data as a Delta Silver table

silver_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_secom_quality"
)

print("Silver table saved successfully.")


Silver table saved successfully.
